# lab2_cnn_finetune (para correr en Kaggle con GPU)

Preprocesamiento (no es un experimento: genera una columna para el modelo tabular).

Ajusta (*fine-tune*) una red de imágenes preentrenada, **EfficientNet-B0** (torchvision, ImageNet), para predecir
**AdoptionSpeed** (regresión) a partir de **hasta 4 fotos por mascota** (`-1.jpg` … `-4.jpg`), combinadas con
*attention pooling* (la red aprende cuánto pesa cada foto).

**Sin fuga:** se usan los **mismos 5 folds** del proyecto (`StratifiedGroupKFold` por `RescuerID`, semilla 42;
se verifica con una huella de `data/folds.csv`). Para cada fold, la red se entrena con los otros 4 y predice el fold
que quedó afuera → predicción *out-of-fold* (OOF) para todo train. Test = promedio de las 5 redes.
Cantidad de vueltas **fija** (no se elige mirando el fold de validación).

**Antes de correr (Kaggle):** Settings → Accelerator **GPU T4 x2** (o P100) e **Internet ON** (para bajar los
pesos de ImageNet, ~20 MB). Agregar como input la competencia `petfinder-adoption-prediction`.
Tarda ~45–60 min con GPU.

**Salida** (en `/kaggle/working/`): `cnn_oof.parquet` con índice `PetID`, columna `cnn_pred` y `conjunto`
(train/test). Mascotas sin foto: vacía. Descargarla y ponerla en `input/lab2-cnn/cnn_oof.parquet`.

In [ ]:
import os, glob, time, hashlib, random
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import torchvision
from torchvision import transforms as T
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import cohen_kappa_score
from scipy.optimize import minimize

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "(¡sin GPU: activar en Settings!)")
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

# Parámetros
N_FOTOS = 4        # máximo de fotos por mascota
TAM = 224          # tamaño de imagen
VUELTAS = 4        # épocas por fold (fijas)
LOTE = 16          # mascotas por lote (hasta 64 imágenes)
LR = 3e-4
WORKERS = os.cpu_count()

In [ ]:
# Rutas: encuentra la competencia en /kaggle/input (layout viejo o nuevo) o en ../input (local)
RAIZ = "/kaggle/input" if os.path.exists("/kaggle/input") else "../input"
csv = glob.glob(f"{RAIZ}/**/train/train.csv", recursive=True)
assert csv, f"No encuentro train/train.csv dentro de {RAIZ}: agregá la competencia petfinder-adoption-prediction como input"
COMP = os.path.dirname(os.path.dirname(csv[0]))
SALIDA = "/kaggle/working" if os.path.exists("/kaggle/working") else "../input/lab2-cnn"
os.makedirs(SALIDA, exist_ok=True)
print("competencia:", COMP)

train = pd.read_csv(f"{COMP}/train/train.csv").set_index("PetID")
test = pd.read_csv(f"{COMP}/test/test.csv").set_index("PetID")

# Mismos folds que el proyecto, verificados con la huella de data/folds.csv
folds = [train.index[v] for _, v in
         StratifiedGroupKFold(5, shuffle=True, random_state=42).split(train, train["AdoptionSpeed"], groups=train["RescuerID"])]
f = pd.concat([pd.Series(k, index=ix) for k, ix in enumerate(folds)]).sort_index()
huella = hashlib.md5("".join(f.index + f.astype(str)).encode()).hexdigest()
assert huella == "e4923328a7291e2ba8f1a12477762d68", f"Los folds NO coinciden con los del proyecto (huella {huella})"
print("Folds verificados (iguales a data/folds.csv):", [len(x) for x in folds])

In [ ]:
# Fotos de cada mascota (-1.jpg, -2.jpg, ...), en orden, hasta N_FOTOS
def fotos_por_mascota(carpeta):
    d = {}
    for ruta in glob.glob(os.path.join(carpeta, "*.jpg")):
        nombre = os.path.basename(ruta)[:-4]
        pet, num = nombre.rsplit("-", 1)          # sufijo real, no strip("-1.jpg")
        d.setdefault(pet, []).append((int(num), ruta))
    return {p: [r for _, r in sorted(v)] for p, v in d.items()}

fotos_tr = fotos_por_mascota(f"{COMP}/train_images")
fotos_te = fotos_por_mascota(f"{COMP}/test_images")
print("mascotas con foto: train", sum(p in fotos_tr for p in train.index), "/", len(train),
      "| test", sum(p in fotos_te for p in test.index), "/", len(test))

NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
AUG = T.Compose([T.RandomResizedCrop(TAM, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                 T.ColorJitter(0.2, 0.2, 0.2), T.ToTensor(), NORM])
FIJA = T.Compose([T.Resize(TAM), T.CenterCrop(TAM), T.ToTensor(), NORM])

def abrir(ruta):
    im = Image.open(ruta)
    im.draft("RGB", (TAM * 2, TAM * 2))          # decodifica JPEG más rápido a tamaño reducido
    return im.convert("RGB")

class Mascotas(torch.utils.data.Dataset):
    def __init__(self, ids, fotos, y=None, entrenar=False):
        self.ids, self.fotos, self.y, self.entrenar = list(ids), fotos, y, entrenar
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, k):
        rutas = self.fotos[self.ids[k]]
        if self.entrenar and len(rutas) > N_FOTOS:
            rutas = [rutas[0]] + random.sample(rutas[1:], N_FOTOS - 1)   # la 1ª siempre, el resto al azar
        rutas = rutas[:N_FOTOS]
        tf = AUG if self.entrenar else FIJA
        x = torch.zeros(N_FOTOS, 3, TAM, TAM)
        m = torch.zeros(N_FOTOS, dtype=torch.bool)
        for j, r in enumerate(rutas):
            x[j], m[j] = tf(abrir(r)), True
        y = torch.tensor(float(self.y[k]) if self.y is not None else 0.0)
        return x, m, y

In [ ]:
# EfficientNet-B0 + attention pooling sobre las fotos de la mascota -> un número (AdoptionSpeed)
class Red(nn.Module):
    def __init__(self):
        super().__init__()
        base = torchvision.models.efficientnet_b0(weights=torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.cuerpo = nn.Sequential(base.features, nn.AdaptiveAvgPool2d(1), nn.Flatten())   # 1280 por foto
        self.atencion = nn.Sequential(nn.Linear(1280, 128), nn.Tanh(), nn.Linear(128, 1))
        self.cabeza = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 1))
    def forward(self, x, m):
        b, n = x.shape[:2]
        h = self.cuerpo(x.flatten(0, 1)).view(b, n, -1)
        a = self.atencion(h).squeeze(-1).masked_fill(~m, -1e4).softmax(dim=1)
        return self.cabeza((a.unsqueeze(-1) * h).sum(1)).squeeze(-1) + 2.5   # centrado en el medio de la escala

def predecir(red, ids, fotos):
    dl = torch.utils.data.DataLoader(Mascotas(ids, fotos), batch_size=LOTE * 2, num_workers=WORKERS)
    red.eval(); salida = []
    with torch.no_grad(), torch.autocast(DEV, enabled=DEV == "cuda"):
        for x, m, _ in dl:
            salida.append(red(x.to(DEV), m.to(DEV)).float().cpu().numpy())
    return np.concatenate(salida)

def entrenar(ids, y):
    red = Red().to(DEV)
    dl = torch.utils.data.DataLoader(Mascotas(ids, fotos_tr, y, entrenar=True), batch_size=LOTE, shuffle=True,
                                     num_workers=WORKERS, drop_last=True,
                                     generator=torch.Generator().manual_seed(SEMILLA))
    opt = torch.optim.AdamW(red.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=VUELTAS * len(dl), pct_start=0.15)
    escala = torch.cuda.amp.GradScaler(enabled=DEV == "cuda")
    for v in range(VUELTAS):
        red.train(); t0, perdidas = time.time(), []
        for x, m, yy in dl:
            with torch.autocast(DEV, enabled=DEV == "cuda"):
                loss = nn.functional.mse_loss(red(x.to(DEV), m.to(DEV)).float(), yy.to(DEV))
            opt.zero_grad(); escala.scale(loss).backward(); escala.step(opt); escala.update(); sched.step()
            perdidas.append(loss.item())
        print(f"   vuelta {v + 1}/{VUELTAS}: MSE train {np.mean(perdidas):.3f} | {(time.time() - t0) / 60:.1f} min")
    return red

In [ ]:
# 5 folds: entrenar con 4, predecir el que queda afuera (OOF) y el test
oof = pd.Series(np.nan, index=train.index)
ids_te = [p for p in test.index if p in fotos_te]
pred_te = []
for k, fold in enumerate(folds):
    t0 = time.time()
    resto = train.index.difference(fold)
    ids = [p for p in resto if p in fotos_tr]
    red = entrenar(ids, train.loc[ids, "AdoptionSpeed"].to_numpy())
    val = [p for p in fold if p in fotos_tr]
    oof.loc[val] = predecir(red, val, fotos_tr)
    pred_te.append(predecir(red, ids_te, fotos_te))
    rmse = np.sqrt(np.mean((oof.loc[val] - train.loc[val, "AdoptionSpeed"]) ** 2))
    print(f"fold {k}: RMSE valid {rmse:.3f} | {(time.time() - t0) / 60:.1f} min")
    del red; torch.cuda.empty_cache()

In [ ]:
# Guardar: OOF de train + promedio de las 5 redes en test
te = pd.Series(np.nan, index=test.index)
te.loc[ids_te] = np.mean(pred_te, axis=0)
res = pd.concat([pd.DataFrame({"cnn_pred": oof, "conjunto": "train"}),
                 pd.DataFrame({"cnn_pred": te, "conjunto": "test"})])
res.index.name = "PetID"
res.to_parquet(os.path.join(SALIDA, "cnn_oof.parquet"))
print(res.groupby("conjunto")["cnn_pred"].describe().round(3))

# Referencia: kappa de la red SOLA (cortes de cada fold elegidos con los otros 4, como en el proyecto)
def cortar(p, c): return np.digitize(p, np.sort(c))
def opt_cortes(p, y):
    return np.sort(minimize(lambda c: -cohen_kappa_score(y, cortar(p, c), weights="quadratic"),
                            (0.5, 1.5, 2.5, 3.5), method="nelder-mead").x)
y = train["AdoptionSpeed"]; ok = oof.notna()
kap = []
for fold in folds:
    v = fold[ok[fold].to_numpy()]; r = ok.index[ok & ~ok.index.isin(fold)]
    kap.append(cohen_kappa_score(y[v], cortar(oof[v], opt_cortes(oof[r], y[r])), weights="quadratic"))
print("Kappa de la red sola por fold:", np.round(kap, 4), "| media", round(float(np.mean(kap)), 4))